# 14 CatBoost Hyperparameter Optimisierung Optuna (on 100%)

## Import

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from optuna_integration import CatBoostPruningCallback

from catboost import CatBoostClassifier

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [ ]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [ ]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)]
    }
)

## Hilfsvariablen

In [ ]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [ ]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_val": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [ ]:
def mv_for_cb(df, fill_var="missing"):
    """df copy mit ersetzten nans"""
    outs = df.copy()
    for col in outs.columns:
        if outs[col].dtype.name == "category":
            if fill_var not in outs[col].cat.categories:
                outs[col] = outs[col].cat.add_categories([fill_var])
            outs[col] = outs[col].fillna(fill_var)
    return outs

In [ ]:
x_train_cb = mv_for_cb(x_train)
x_val_cb = mv_for_cb(x_val)
x_test_cb = mv_for_cb(x_test)
x_full_cb = mv_for_cb(x_full)

x_train_no_calc_cb = x_train_cb[feat_cols_no_calc]
x_val_no_calc_cb = x_val_cb[feat_cols_no_calc]
x_test_no_calc_cb = x_test_cb[feat_cols_no_calc]
x_full_no_calc_cb = x_full_cb[feat_cols_no_calc]

pd.Series(
    {
        "train": [len(x_train_cb), len(x_train_no_calc_cb)],
        "val": [len(x_val_cb), len(x_val_no_calc_cb)],
        "test": [len(x_test_cb), len(x_test_no_calc_cb)],
        "full": [len(x_full_cb), len(x_full_no_calc_cb)]
    }
)

In [ ]:
def subsample(x_set, y_set, anteil, random_state=RANDOM_STATE):
    """Subsample erzeugen"""
    if anteil >= 1.0:
        return x_set, y_set
    rng = np.random.default_rng(random_state)
    n = int(len(x_set) * anteil)
    idx_sub = rng.permutation(len(x_set))[:n]
    return x_set.iloc[idx_sub], y_set.iloc[idx_sub]

In [ ]:
x_sub_75, y_sub_75 = subsample(x_train_cb, y_train, 0.75)
x_sub_no_calc_75, y_sub_no_calc_75 = subsample(x_train_no_calc_cb, y_train, 0.75)

pd.Series(
    {
        "sub": len(x_sub_75),
        "sub no calc": len(x_sub_no_calc_75),
        "anteil target": y_sub_75.mean()
    }
)

## Optuna Hilfe

## Suchräume

|Paramerter|Bereich|
|---|---|
|learning rate|0.01-0.3|
|depth|4-10|
|l2_leaf_reg|1-30|
|random_strength|1e3-10|
|bootstrap_type|Bayesian / Bernoulli / MVS|
|bagging_temperature|0-5|
|one_hot_max_size|2-105|
|leaf_estimation_iterations|1-10|
|auto_class_weights|none / Balanced|

In [ ]:
fixed_params_plain = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Plain"
}

In [ ]:
fixed_params_ordered = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Ordered"
}

In [ ]:
def suchraum_params(trial):
    """Suchraum definition"""
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "one_hot_max_size": trial.suggest_categorical("one_hot_max_size", [2, 10, 18, 105]),
        "leaf_estimation_iterations": trial.suggest_int("leaf_estimation_iterations", 1, 10),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", [None, "Balanced"]),
        "bootstrap_type": trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
    }

    if params["bootstrap_type"] == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.0, 5.0)
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

    return params

## Plain mit Calc

In [ ]:
def objective_plain_calc_full(trial):
    modell = CatBoostClassifier(
        **fixed_params_plain,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_train_cb, y_train, eval_set=[(x_val_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_cb)[:, 1])

In [ ]:
study_plain_with_calc_full = optuna.create_study(
    study_name = "CatBoost_Plain_with_calc_full",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

In [ ]:
study_plain_with_calc_full.optimize(
    objective_plain_calc_full,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.DataFrame({
    "best auc_val": study_plain_with_calc_full.best_value,
    "best gini": 2 * study_plain_with_calc_full.best_value - 1,
    "trials": len(study_plain_with_calc_full.trials),
    "pruned": len([t for t in study_plain_with_calc_full.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_plain_with_calc_full.trials if t.state.name == "FAIL"]),
    "best params": study_plain_with_calc_full.best_params
})

## Plain ohne Calc

In [ ]:
def objective_plain_no_calc_full(trial):
    modell = CatBoostClassifier(
        **fixed_params_plain,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_train_no_calc_cb, y_train, eval_set=[(x_val_no_calc_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_cb)[:, 1])

In [ ]:
study_plain_without_calc_full = optuna.create_study(
    study_name = "CatBoost_Plain_without_calc_full",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

In [ ]:
study_plain_without_calc_full.optimize(
    objective_plain_no_calc_full,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.Series({
    "best auc_val": study_plain_without_calc_full.best_value,
    "best gini": 2 * study_plain_without_calc_full.best_value - 1,
    "trials": len(study_plain_without_calc_full.trials),
    "pruned": len([t for t in study_plain_without_calc_full.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_plain_without_calc_full.trials if t.state.name == "FAIL"]),
    "best params": study_plain_without_calc_full.best_params
})

In [ ]:
study_plain_without_calc_full.best_params

## Ordered ohne Calc

In [ ]:
def objective_ordered_no_calc_full(trial):
    modell = CatBoostClassifier(
        **fixed_params_ordered,
        **suchraum_params(trial)
    )
    pruning_callback = CatBoostPruningCallback(trial, "AUC")

    modell.fit(x_train_no_calc_cb, y_train, eval_set=[(x_val_no_calc_cb, y_val)], callbacks=[pruning_callback])
    pruning_callback.check_pruned()

    return roc_auc_score(y_val, modell.predict_proba(x_val_no_calc_cb)[:, 1])

In [ ]:
study_ordered_without_calc_full = optuna.create_study(
    study_name = "CatBoost_Ordered_without_calc_full",
    storage = "sqlite:///catboost_opti.db",
    load_if_exists = True,
    direction = "maximize",
    sampler = TPESampler(seed=RANDOM_STATE),
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=200)
)

In [ ]:
study_ordered_without_calc_full.optimize(
    objective_ordered_no_calc_full,
    n_trials = 50,
    timeout = 4*60*60,
    catch = (Exception, )
)

In [ ]:
pd.Series({
    "best auc_val": study_ordered_without_calc_full.best_value,
    "best gini": 2 * study_ordered_without_calc_full.best_value - 1,
    "trials": len(study_ordered_without_calc_full.trials),
    "pruned": len([t for t in study_ordered_without_calc_full.trials if t.state.name == "PRUNED"]),
    "fail": len([t for t in study_ordered_without_calc_full.trials if t.state.name == "FAIL"]),
    "best params": study_ordered_without_calc_full.best_params
})

In [ ]:
study_ordered_without_calc_full.best_params

## Load Study

In [ ]:
study_plain_with_calc_full_loaded = optuna.load_study(
    study_name = "CatBoost_Plain_with_calc_full",
    storage = "sqlite:///catboost_opti.db"
)


In [ ]:
study_plain_without_calc_full_loaded = optuna.load_study(
    study_name = "CatBoost_Plain_without_calc_full",
    storage = "sqlite:///catboost_opti.db"
)

In [ ]:
study_ordered_without_calc_full_loaded = optuna.load_study(
    study_name = "CatBoost_Ordered_without_calc_full",
    storage = "sqlite:///catboost_opti.db"
)

## Best Params again

In [ ]:
study_plain_with_calc_full_loaded.best_params

In [ ]:
study_plain_without_calc_full_loaded.best_params

In [ ]:
study_ordered_without_calc_full_loaded.best_params

## 100% Train

In [ ]:
results = []
train_times = {}

In [ ]:
name = "L_cb_opt_01_full"

L_cb_opt_01_full = CatBoostClassifier(
    **fixed_params_plain,
    **study_plain_with_calc_full_loaded.best_params
)

start = time.time()
L_cb_opt_01_full.fit(
    x_train_cb, y_train, 
    eval_set=(x_val_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_01_full, x_train_cb, y_train, x_val_cb, y_val, train_times[name], best_iter=L_cb_opt_01_full.get_best_iteration())
)

In [ ]:
name = "L_cb_opt_02_full"

L_cb_opt_02_full = CatBoostClassifier(
    **fixed_params_plain,
    **study_plain_without_calc_full_loaded.best_params
)

start = time.time()
L_cb_opt_02_full.fit(
    x_train_no_calc_cb, y_train, 
    eval_set=(x_val_no_calc_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_02_full, x_train_no_calc_cb, y_train, x_val_no_calc_cb, y_val, train_times[name], best_iter=L_cb_opt_02_full.get_best_iteration())
)

In [ ]:
name = "L_cb_opt_03_full"

L_cb_opt_03_full = CatBoostClassifier(
    **fixed_params_ordered,
    **study_ordered_without_calc_full_loaded.best_params
)

start = time.time()
L_cb_opt_03_full.fit(
    x_train_no_calc_cb, y_train, 
    eval_set=(x_val_no_calc_cb, y_val), 
    verbose=250
)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_cb_opt_03_full, x_train_no_calc_cb, y_train, x_val_no_calc_cb, y_val, train_times[name], best_iter=L_cb_opt_03_full.get_best_iteration())
)

In [ ]:
pd.DataFrame(results)

## Save Models

In [ ]:
Modelle = [
    (L_cb_opt_01_full, "L_cb_opt_01_full"),
    (L_cb_opt_02_full, "L_cb_opt_02_full"),
    (L_cb_opt_03_full, "L_cb_opt_03_full")
]

for modell, name in Modelle:
    modell.save_model(f"{name}.cbm")

## Notizen
- Die Learning Curves zeigten kein definitives Plateau, also optimiere ich hier nochmal anhand des vollen train sets
- ich setze die zeitbegrenzung mal auf 4 stunden zur sicherheit


